<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Powerpredict" data-toc-modified-id="Powerpredict-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Powerpredict</a></span><ul class="toc-item"><li><span><a href="#Loading-the-dataset" data-toc-modified-id="Loading-the-dataset-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>Loading the dataset</a></span><ul class="toc-item"><li><span><a href="#Loading-the-models" data-toc-modified-id="Loading-the-models-1.1.1"><span class="toc-item-num">1.1.1&nbsp;&nbsp;</span>Loading the models</a></span></li></ul></li></ul></li></ul></div>

# Powerpredict

Daily power consumption is related to the weather (rain, sunshine, temperature, etc). 
Prediction of the power consumption based on the weather is relevant for energy suppliers.
In this dataset you have to use the provided weather information to predict the power consumption.

## Loading the dataset

In [2]:
import pandas as pd
import os

In [3]:
DATASET_PATH = "."
if os.path.exists("/data/mlproject22"):
    DATASET_PATH = "/data/mlproject22"

In [4]:
powerpredict = pd.read_csv(os.path.join(DATASET_PATH,"powerpredict.csv.zip"))
X = powerpredict.drop(columns=["power_consumption"])
y = powerpredict[["power_consumption"]]

print(X.shape)
print(y.shape)

(39997, 65)
(39997, 1)


Some columns are dropped here for simplicity, but they might provide useful information as well, so you might want to use them.

In [5]:
def drop_object_columns(df):
    drop_cols = [c for t, c in zip([t !="object" for t in df.dtypes], df.columns) if not t]
    return df.drop(columns=drop_cols)

DOC = drop_object_columns

def predict_show_metrics(name, reg, metric):
    print(f"{name}", metric(y, reg.predict(DOC(X))))


Here is an example dummy ML method showing the success of such a simple predictor:

In [6]:
import sklearn.metrics
import sklearn.dummy
reg = sklearn.dummy.DummyRegressor()
reg.fit(DOC(X), y)
metric = sklearn.metrics.mean_absolute_error

predict_show_metrics("Dummy", reg, metric)

Dummy 4085.9953308309014


### Loading the models

In [37]:
import joblib
import torch
import torch.nn as nn

# Loads the pre-trained Random Forest model
best_rf_model = joblib.load('model_rf.pkl')
model_large = joblib.load('model_rf_large.pkl')
print("Model large:")
print("Random Forest Parameters:")
print(f"Number of Estimators: {model_large.n_estimators}")
print(f"Max Depth: {model_large.max_depth}")
print(f"Min Samples Split: {model_large.min_samples_split}")
print(f"Min Samples Leaf: {model_large.min_samples_leaf}")
      
print("Model small:")
print("Random Forest Parameters:")
print(f"Number of Estimators: {best_rf_model.n_estimators}")
print(f"Max Depth: {best_rf_model.max_depth}")
print(f"Min Samples Split: {best_rf_model.min_samples_split}")
print(f"Min Samples Leaf: {best_rf_model.min_samples_leaf}")


# Load the pre-trained scaler and pca
# Dont use the scaler and the pca for the final score, because also not used in the get_score function!
# scaler = joblib.load('scaler.pkl')
# pca = joblib.load('pca.pkl')

# Ensure all data are numeric
X = DOC(X)

# Load the pre-trained Random Forest model
# best_rf_model = joblib.load('model_rf_large.pkl')


# Load the pre-trained nn model state_dict
state_dict = torch.load('model_nn.pth')

print("\nNeural Network Parameters:")
print(f"Input Size: {input_size}")
print(f"Hidden Layer 1 Size: {best_nn_model.fc1.out_features}")
print(f"Hidden Layer 2 Size: {best_nn_model.fc2.out_features}")

# Define the Neural Network class with the correct architecture
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_size1=32, hidden_size2=64):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, 1)  # Output a single value for regression
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Determine the input size based on the number of features in X
input_size = X.shape[1]

best_nn_model = NeuralNetwork(input_size, hidden_size1=32, hidden_size2=64)
best_nn_model.load_state_dict(state_dict)
best_nn_model.eval()

# Set device to CPU
device = torch.device('cpu')
best_nn_model.to(device)

# Prepare the data for PyTorch
X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1, 1).to(device)

/opt/conda/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.5.0 when using version 1.4.1.post1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.5.0 when using version 1.4.1.post1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model large:
Random Forest Parameters:
Number of Estimators: 200
Max Depth: 30
Min Samples Split: 2
Min Samples Leaf: 1
Model small:
Random Forest Parameters:
Number of Estimators: 150
Max Depth: 20
Min Samples Split: 5
Min Samples Leaf: 2

Neural Network Parameters:
Input Size: 55
Hidden Layer 1 Size: 32
Hidden Layer 2 Size: 64


You have to implement a simple method that performs the predictions with the given signature:

In [32]:
def leader_board_predict_fn(values):
    values_scaled = scaler.transform(values)
    values_tensor = torch.tensor(values_scaled, dtype=torch.float32).to(device)
    
    # Neural Network prediction
    with torch.no_grad():
        nn_predictions = best_nn_model(values_tensor).cpu().numpy().flatten()
    
    # Random Forest prediction
    rf_predictions = best_rf_model.predict(values_scaled)

    return nn_predictions, rf_predictions  # or return nn_predictions


which will then be used to calculate the leaderboard score in a way similar to this:

In [33]:
 # Changed: all output printed on the screen due to file-store error
def get_score():
    """
    Function to compute scores for train and test datasets.
    """
    
    import sklearn.metrics
    import pandas as pd
    import pathlib
    import os

    try:
        TEST_DATASET_PATH = "."
        if os.path.exists("/data/mlproject22/"):
            TEST_DATASET_PATH = "/data/mlproject22/"

        test_data = pd.read_csv(os.path.join(TEST_DATASET_PATH, "powerpredict.csv.zip"))
        X_test = test_data.drop(columns=["power_consumption"])
        X_test = DOC(X_test) # Also ensured that non-numerical objects are dropped
        y_test = test_data[["power_consumption"]]
        nn_pred, rf_pred = leader_board_predict_fn(X_test)
        nn_dataset_score = sklearn.metrics.mean_absolute_error(y_test, nn_pred)
        rf_dataset_score = sklearn.metrics.mean_absolute_error(y_test, rf_pred)
    except Exception as e:
        nn_dataset_score = float("nan")
        rf_dataset_score = float("nan")
        print(f"Error calculating train dataset score: {e}")
    print(f"Train Dataset Score (NN): {nn_dataset_score}")
    print(f"Train Dataset Score (RF): {rf_dataset_score}")

    import os
    import pwd
    import time
    import datetime
    import pandas as pd
    user_id = pwd.getpwuid(os.getuid()).pw_name
    curtime = time.time()
    dt_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

    try:  
        HIDDEN_DATASET_PATH = os.path.expanduser("/data/mlproject22-test-data/")
        test_data = pd.read_csv(os.path.join(HIDDEN_DATASET_PATH, "hidden_powerpredict.csv.zip"))
        X_test = test_data.drop(columns=["power_consumption"])
        X_test = DOC(X_test) # Also ensured that non-numerical objects are dropped
        y_test = test_data[["power_consumption"]]
        nn_pred, rf_pred = leader_board_predict_fn(X_test)
        nn_hiddendataset_score = sklearn.metrics.mean_absolute_error(y_test, nn_pred)
        rf_hiddendataset_score = sklearn.metrics.mean_absolute_error(y_test, rf_pred)
        print(f"Test Dataset Score (NN): {nn_hiddendataset_score}")
        print(f"Test Dataset Score (RF): {rf_hiddendataset_score}")
        score_dict = dict(
            nn_score_hidden=nn_hiddendataset_score,
            nn_score_train=nn_dataset_score,
            rf_score_hidden=rf_hiddendataset_score,
            rf_score_train=rf_dataset_score,
            unixtime=curtime,
            user=user_id,
            dt=dt_now,
            comment="",
        )
    except Exception as e:
        err = str(e)
        score_dict = dict(
            nn_score_hidden=float("nan"),
            nn_score_train=nn_dataset_score,
            rf_score_hidden=float("nan"),
            rf_score_train=rf_dataset_score,
            unixtime=curtime,
            user=user_id,
            dt=dt_now,
            comment=err
        )
        print(f"Error calculating hidden dataset score: {err}")

    pd.DataFrame([score_dict]).to_csv("powerpredict.csv", index=False)

get_score()


/opt/conda/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


Train Dataset Score (NN): 28399.40258022413
Train Dataset Score (RF): 4427.538848974739
Test Dataset Score (NN): 28651.189358883756
Test Dataset Score (RF): 4184.902301747096


/opt/conda/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
